## Middleware

- These are basically hooks which can act as a trigger for a tool call or a function.
- Can be used to control what happens inside an agent.
- The hooks can be retries, fall backs, rate limits, tranforming prompts, tool selection and output formatting etc.


In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

There are few in-built hooks too like 
- Summarization
- Human In Feedback
... more in Langchain

### Summarization
automatically summarize when approaching token limits

In [2]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage

agent = create_agent(
    model="groq:qwen/qwen3.6-27b",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:groq/compound-mini",  # We can use flash or mini models for summarization to save tokens
            trigger=("messages", 10),  # At every 10 messages summarization will work.
            keep=("messages", 4),  # To have enough context keep last 4 chats
        )
    ],
)

### What is a thread?

A thread is one conversation, identified by a `thread_id`. It is the key the `checkpointer` stores state under.

- Same `thread_id`: loads the saved messages, appends the new turn, saves again. This is what makes the agent remember.
- New `thread_id`: fresh conversation, empty state.
- No `config`: error, since the checkpointer has no key to save under.

The agent itself is stateless. Memory lives in the checkpointer, so one agent can serve many conversations (one `thread_id` per user).

`InMemorySaver` keeps state in RAM, so it is lost when the kernel restarts. Use `SqliteSaver` or `PostgresSaver` to persist.

In [3]:
config = {"configurable": {"thread_id": "testing-01"}}

In [4]:
import re
from IPython.display import Markdown, display


def clean(text):
    """qwen wraps its reasoning in <think> tags, which Markdown hides as HTML."""
    return re.sub(r"<think>.*?</think>", "", text, flags=re.S).strip()


def get_summary(messages):
    """The middleware injects the summary as a HumanMessage tagged lc_source."""
    return next(
        (m.content for m in messages if m.additional_kwargs.get("lc_source") == "summarization"),
        None,
    )


def ask(agent, questions, config):
    last_summary = None
    for q in questions:
        messages = agent.invoke({"messages": [HumanMessage(content=q)]}, config)["messages"]

        display(Markdown(f"**Q:** {q}"))
        display(Markdown(clean(messages[-1].content)))

        summary = get_summary(messages)
        if summary and summary != last_summary:
            display(Markdown(f"**Summary injected:**\n\n{clean(summary)}"))
            last_summary = summary

        print(f"{len(messages)} messages in state\n")


questions_list = [
    "In one sentence, what is a vector database?",
    "Name two popular ones.",
    "Which one is easiest to run locally?",
    "How is it different from a normal SQL database?",
    "What is an embedding?",
    "Which model would you use to create embeddings?",
    "What does cosine similarity measure?",
    "Now, what was the very first thing I asked you about?",
]

ask(agent, questions_list, config)

**Q:** In one sentence, what is a vector database?

A vector database is a specialized storage system designed to efficiently store, index, and query high-dimensional numerical vectors (embeddings) to enable fast similarity searches for AI and machine learning applications.

2 messages in state



**Q:** Name two popular ones.

Two popular vector databases are **Pinecone** and **Milvus**.

...

Cosine similarity measures the **cosine of the angle** between two vectors. In embedding space, it tells you how aligned the *direction* (semantic meaning) of two vectors is, while completely ignoring their *magnitude* (length or intensity).

**Key points:**
- **Range:** `-1` to `1` (most text embeddings are pre-normalized to unit length, so you'll typically see `0`–`1`)
- `1.0` = identical direction (maximum similarity)
- `0.0` = orthogonal (no semantic relationship)
- `-1.0` = opposite direction (rare for modern text embeddings)
- **Formula:** `(A · B) / (||A|| × ||B||)` → dot product divided by the product of their magnitudes

**Why it’s the standard for embeddings:**  
Embedding models are trained to place semantically related content in the same *direction*, regardless of document length, vocabulary size, or token count. Cosine similarity captures exactly that—pure semantic alignment—making it the default metric in nearly all vector databases and AI search pipelines. (Note: Because most modern embeddings are normalized to length `1`, cosine similarity becomes mathematically identical to a simple dot product, which speeds up computation.)

Want a quick Python example showing how it's calculated, or how it compares to Euclidean distance in practice?

8 messages in state



**Q:** Now, what was the very first thing I asked you about?

Based on the session context provided at the start, your very first question was about **vector databases**—specifically asking what they are, popular examples, which can be run locally, and how they differ from traditional SQL databases.

10 messages in state



In [5]:
# What is stored in this thread right now. No API call.
for m in agent.get_state(config).values["messages"]:
    tag = m.additional_kwargs.get("lc_source", "")
    print(f"{type(m).__name__:12} {tag:14} {clean(m.content)[:70]!r}")

HumanMessage summarization  'Here is a summary of the conversation to date:\n\n## SESSION INTENT\nThe '
AIMessage                   'A traditional SQL database stores structured data in tables and is opt'
HumanMessage                'What is an embedding?'
AIMessage                   'An embedding is a numerical representation of data (like text, images,'
HumanMessage                'Which model would you use to create embeddings?'
AIMessage                   'There’s no single “best” model, but the right choice depends on your d'
HumanMessage                'What does cosine similarity measure?'
AIMessage                   'Cosine similarity measures the **cosine of the angle** between two vec'
HumanMessage                'Now, what was the very first thing I asked you about?'
AIMessage                   'Based on the session context provided at the start, your very first qu'


In [6]:
# We can also trigger on tokens instead of message count.

token_agent = create_agent(
    model="groq:qwen/qwen3.6-27b",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:groq/compound-mini",
            trigger=("tokens", 500),  # Summarize once history passes 500 tokens.
            keep=("tokens", 200),  # Keep roughly the last 200 tokens.
        )
    ],
)

token_config = {"configurable": {"thread_id": "tokens-01"}}

ask(token_agent, questions_list[:5], token_config)

**Q:** In one sentence, what is a vector database?

A vector database is a specialized storage system that indexes and queries high-dimensional numerical embeddings of data to enable fast similarity searches and semantic matching for AI and machine learning applications.

2 messages in state



**Q:** Name two popular ones.

Two popular vector databases are **Pinecone** and **Weaviate**.

...

3 messages in state



**Q:** What is an embedding?

An **embedding** is a dense, fixed-length numerical vector (a list of floating-point numbers) that represents a piece of data in a way that captures its meaning, features, or relationships. Instead of using sparse or manual representations, embeddings are learned by machine learning models to place semantically or structurally similar items close together in a multi-dimensional space.

### 🔑 Key Ideas
- **From discrete to continuous**: Text, images, audio, graph nodes, or user IDs are converted into vectors of numbers (e.g., `[0.23, -0.81, 0.45, ...]`).
- **Semantic proximity**: Similar concepts end up near each other. For text, `"dog"` and `"puppy"` will have similar vectors; `"king"` and `"queen"` will be close, and mathematical relationships can even emerge (`king - man + woman ≈ queen`).
- **Dense & efficient**: Unlike one-hot encoding (which creates huge, mostly-zero vectors), embeddings are compact and highly informative, making them ideal for neural networks and fast similarity search.
- **Model-learned**: Embeddings aren't handcrafted. They're trained on large datasets using architectures like Word2Vec, BERT, CLIP, or modern LLMs, which adjust the numbers to minimize prediction or contrastive loss.

### 🛠️ Common Uses
- **Semantic search & retrieval**: Find documents/passages by meaning, not just keywords
- **Recommendation systems**: Match users to items based on latent preferences
- **Multimodal AI**: Align text and images in the same space (e.g., CLIP)
- **Vector databases**: Store and query embeddings efficiently for nearest-neighbor search (ties directly to your earlier question about alternatives to SQL)

### 📏 Practical Notes
- Embeddings have a fixed dimensionality (e.g., 384, 768, 1536, or 4096), chosen to balance expressiveness vs. compute/storage cost.
- Similarity is typically measured with cosine similarity, dot product, or Euclidean distance.
- Quality depends heavily on the training data and architecture; domain-specific embeddings often outperform generic ones.

If you're working with a specific use case (search, RAG, recommendations, etc.), I can suggest embedding models, dimension choices, or how to pair them with vector databases.

**Summary injected:**

Here is a summary of the conversation to date:

## SESSION INTENT
User wants to extract the most relevant context from the conversation history to replace it, freeing up token space.

## SUMMARY
- The system hit a rate‑limit error while attempting to generate a prior summary (error code 429, tokens‑per‑minute limit reached).
- User asked: “How is it different from a normal SQL database?” (the referent “it” was unclear due to missing earlier context).
- Assistant responded by:
  * Acknowledging the missing context and asking for clarification.
  * Providing a concise comparison of common alternatives to traditional SQL databases:
    - **NoSQL** (MongoDB, Cassandra, DynamoDB): schema‑less, horizontal scaling, no joins.
    - **Vector databases** (Pinecone, Weaviate, pgvector): store high‑dimensional vectors, optimized for similarity/semantic search.
    - **Graph databases** (Neo4j, Amazon Neptune): node/edge model, efficient relationship traversal.
    - **Time‑Series databases** (InfluxDB, TimescaleDB): optimized for timestamped data, fast ingestion and time‑window queries.
    - **NewSQL** (CockroachDB, Yugabyte): relational/ACID semantics with distributed scalability.
  * Highlighting traditional SQL strengths: strict schemas, ACID compliance, powerful `JOIN`s, mature tooling.
  * Requesting the user specify the exact system for a precise side‑by‑side comparison.

## ARTIFACTS
None.

## NEXT STEPS
No further actions needed; the extracted summary can replace the original conversation history.

3 messages in state

